# τέλος Unified 3-Paradigm Training Suite

This notebook coordinates and executes sequential training under identical token budgets and hyperparameter parity across all three foundational paradigms:
1. **Autoregressive (AR) Baseline** — Causal attention, standard left-to-right next-token prediction.
2. **Masked Discrete Diffusion (MDLM)** — Bidirectional attention, continuous absorbing-state ($[\text{MASK}]$) diffusion with ELBO $1/t$ loss.
3. **Uniform Noise Diffusion (UNDLM)** — Bidirectional attention, continuous discrete uniform vocabulary corruption with $1/t$ reweighted loss.

All three models share identical transformer parameters (RoPE, SwiGLU, RMSNorm, Weight Tying) and exact data ordering for fair, controlled benchmarking.

In [ ]:
import os
import sys
import time
import gc
import yaml
from pathlib import Path

# Set project root
project_root = Path.cwd()
while not (project_root / "mdiff").exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
from mdiff.model.mlx_components import MLXTelosTransformer, load_upscaled_weights
from mdiff.training.trainer import TelosMLXTrainer as MDLM_Trainer
from undiff.training.trainer import TelosMLXUNDLMTrainer as UNDLM_Trainer
from ar.model.mlx_components import MLXCausalTransformer
from ar.training.trainer import TelosMLXARTrainer as AR_Trainer


def run_unified_training_suite(config_path, upscale_from_tier=None):
    """
    Executes unified sequential training across AR, MDLM, and UNDLM paradigms.
    
    Args:
        config_path: Relative or absolute path to the target YAML config file.
        upscale_from_tier: Optional source tier (e.g. '12m') to load pretrained
                           weights from, mapping depths and zero-padding dimensions.
    """
    with open(config_path, "r") as f:
        base_cfg = yaml.safe_load(f)
    
    cfg_stem = Path(config_path).stem
    tier = "25m" if "25m" in cfg_stem else ("50m" if "50m" in cfg_stem else "12m")
    save_every = base_cfg.get("checkpoint", {}).get("save_every_steps", 50)
    
    suite_start = time.time()
    mode_tag = f" [UPSCALING FROM {upscale_from_tier.upper()}]" if upscale_from_tier else " [TRAIN FROM SCRATCH]"
    print("=" * 90)
    print(f"STARTING UNIFIED 3-PARADIGM SUITE: {cfg_stem} ({tier.upper()}){mode_tag}")
    print(f"Steps: {base_cfg.get('training', {}).get('max_steps')} | Batch: {base_cfg.get('training', {}).get('batch_size')} | Grad Accum: {base_cfg.get('training', {}).get('gradient_accumulation')}")
    print("=" * 90)
    
    # -------------------------------------------------------------------------
    # PARADIGM 1: Autoregressive Baseline (AR)
    # -------------------------------------------------------------------------
    print("\n>>> PARADIGM 1/3: Autoregressive (AR) Baseline <<<")
    ar_cfg = yaml.safe_load(yaml.dump(base_cfg))
    ar_cfg["checkpoint"] = {"dir": f"checkpoints/ar/{tier}/{cfg_stem}", "save_every_steps": save_every}
    
    ar_model = MLXCausalTransformer(**ar_cfg["model"])
    if upscale_from_tier:
        src_stem = cfg_stem.replace(tier, upscale_from_tier)
        src_ckpt = f"checkpoints/ar/{upscale_from_tier}/{src_stem}/model.safetensors"
        src_cfg_path = f"configs/unified/{upscale_from_tier}/{src_stem}.yaml"
        print(f"  -> Initializing AR model with upscaled weights from {src_ckpt}")
        load_upscaled_weights(ar_model, ar_cfg["model"], src_ckpt, src_cfg_path)
        
    ar_model.set_dtype(mx.bfloat16)
    ar_trainer = AR_Trainer(ar_model, ar_cfg)
    ar_trainer.train()
    del ar_model, ar_trainer
    gc.collect()
    mx.clear_cache()
    
    # -------------------------------------------------------------------------
    # PARADIGM 2: Masked Discrete Diffusion (MDLM)
    # -------------------------------------------------------------------------
    print("\n>>> PARADIGM 2/3: Masked Discrete Diffusion (MDLM) <<<")
    mdlm_cfg = yaml.safe_load(yaml.dump(base_cfg))
    mdlm_cfg["checkpoint"] = {"dir": f"checkpoints/masked/{tier}/{cfg_stem}", "save_every_steps": save_every}
    
    mdlm_model = MLXTelosTransformer(**mdlm_cfg["model"])
    if upscale_from_tier:
        src_stem = cfg_stem.replace(tier, upscale_from_tier)
        src_ckpt = f"checkpoints/masked/{upscale_from_tier}/{src_stem}/model.safetensors"
        src_cfg_path = f"configs/unified/{upscale_from_tier}/{src_stem}.yaml"
        print(f"  -> Initializing MDLM model with upscaled weights from {src_ckpt}")
        load_upscaled_weights(mdlm_model, mdlm_cfg["model"], src_ckpt, src_cfg_path)
        
    mdlm_model.set_dtype(mx.bfloat16)
    mdlm_trainer = MDLM_Trainer(mdlm_model, mdlm_cfg)
    mdlm_trainer.train()
    del mdlm_model, mdlm_trainer
    gc.collect()
    mx.clear_cache()
    
    # -------------------------------------------------------------------------
    # PARADIGM 3: Uniform Noise Diffusion (UNDLM)
    # -------------------------------------------------------------------------
    print("\n>>> PARADIGM 3/3: Uniform Noise Diffusion (UNDLM) <<<")
    undlm_cfg = yaml.safe_load(yaml.dump(base_cfg))
    undlm_cfg["checkpoint"] = {"dir": f"checkpoints/uniform/{tier}/{cfg_stem}", "save_every_steps": save_every}
    
    undlm_model = MLXTelosTransformer(**undlm_cfg["model"])
    if upscale_from_tier:
        src_stem = cfg_stem.replace(tier, upscale_from_tier)
        src_ckpt = f"checkpoints/uniform/{upscale_from_tier}/{src_stem}/model.safetensors"
        src_cfg_path = f"configs/unified/{upscale_from_tier}/{src_stem}.yaml"
        print(f"  -> Initializing UNDLM model with upscaled weights from {src_ckpt}")
        load_upscaled_weights(undlm_model, undlm_cfg["model"], src_ckpt, src_cfg_path)
        
    undlm_model.set_dtype(mx.bfloat16)
    undlm_trainer = UNDLM_Trainer(undlm_model, undlm_cfg)
    undlm_trainer.train()
    del undlm_model, undlm_trainer
    gc.collect()
    mx.clear_cache()
    
    elapsed_hours = (time.time() - suite_start) / 3600.0
    print("=" * 90)
    print(f"COMPLETED SUITE FOR {cfg_stem} IN {elapsed_hours:.2f} HOURS!")
    print("=" * 90)


## 1. 12.5M Scale Study (Train-From-Scratch)
Architecture: $d=256$, $L=13$, $\text{heads}=4$, $\text{seq\_len}=512$ (~12.54M parameters).

In [ ]:
# 12.5M 1:1 Ratio (~12.5M tokens, 96 steps)
run_unified_training_suite("configs/unified/12m/telos_12m_r1.yaml")
# 12.5M 1:5 Ratio (~62.5M tokens, 480 steps)
run_unified_training_suite("configs/unified/12m/telos_12m_r5.yaml")
# 12.5M 1:10 Ratio (~125M tokens, 960 steps)
run_unified_training_suite("configs/unified/12m/telos_12m_r10.yaml")
# 12.5M 1:15 Ratio (~188M tokens, 1440 steps)
run_unified_training_suite("configs/unified/12m/telos_12m_r15.yaml")
# 12.5M 1:20 Ratio (~250M tokens, 1920 steps)
run_unified_training_suite("configs/unified/12m/telos_12m_r20.yaml")
# 12.5M 1:25 Ratio (~314M tokens, 2400 steps)
run_unified_training_suite("configs/unified/12m/telos_12m_r25.yaml")
# 12.5M 1:30 Ratio (~376M tokens, 2880 steps)
run_unified_training_suite("configs/unified/12m/telos_12m_r30.yaml")

STARTING UNIFIED 3-PARADIGM SUITE FOR: telos_12m_r1 (Tier: 12m)
Max Steps: 96 | Batch Size: 32 | Grad Accum: 8

>>> PARADIGM 1/3: Autoregressive (AR) Baseline <<<
  Loading pre-tokenized dataset from data/python_corpus_mac.bin...
  Checkpoint Directory: checkpoints/ar/12m/telos_12m_r1 (AR Canonical)


KeyboardInterrupt: 

## 2. 25M Upscaling Suite (12.5M $\to$ 25M)
Initializes target 25M architectures ($d=512$, $L=8$, $\text{heads}=8$) from corresponding 12.5M checkpoints ($d=256$, $L=13$, $\text{heads}=4$) via depth interpolation and zero-padded width expansions across AR, MDLM, and UNDLM.

In [ ]:
# [1/4] 25M 1:1 Ratio (~25M tokens, 191 steps) — Upscaled from 12.5M 1:1
run_unified_training_suite("configs/unified/25m/telos_25m_r1.yaml", upscale_from_tier="12m")

In [ ]:
# [2/4] 25M 1:10 Ratio (~250M tokens, 1910 steps) — Upscaled from 12.5M 1:10
run_unified_training_suite("configs/unified/25m/telos_25m_r10.yaml", upscale_from_tier="12m")

In [ ]:
# [3/4] 25M 1:20 Ratio (~500M tokens, 3820 steps) — Upscaled from 12.5M 1:20
run_unified_training_suite("configs/unified/25m/telos_25m_r20.yaml", upscale_from_tier="12m")

In [ ]:
# [4/4] 25M 1:25 Ratio (~625M tokens, 4775 steps) — Upscaled from 12.5M 1:25
run_unified_training_suite("configs/unified/25m/telos_25m_r25.yaml", upscale_from_tier="12m")

## 3. 25M Train-From-Scratch Suite (1:30, 1:35)
Trains 25M models from cold random initialization under high token over-training budgets (1:30 and 1:35 ratios).

In [ ]:
# [1/2] 25M 1:30 Ratio (~750M tokens, 5730 steps) — Train From Scratch
run_unified_training_suite("configs/unified/25m/telos_25m_r30.yaml")

In [ ]:
# [2/2] 25M 1:35 Ratio (~875M tokens, 6685 steps) — Train From Scratch
run_unified_training_suite("configs/unified/25m/telos_25m_r35.yaml")